# Historical Customer Value Analysis

## Business Objective

Understand how much revenue customers generate across their observed purchasing history, and assess whether initial purchase value and repeat-purchase propensity provide meaningful signals of longer-term customer value.

This analysis will establish a historical revenue baseline before developing predictive Customer Lifetime Value.

## Data Foundation

Identify the transaction-level data available for customer revenue measurement, including orders, order items, payments, and customer identifiers.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

# Locate the project data directory
project_root = Path.cwd().parent
data_directory = project_root / "data"

# Inspect available data files before selecting sources
for file_path in sorted(data_directory.rglob("*")):
    if file_path.is_file():
        print(file_path.relative_to(project_root))

data\.gitkeep
data\olist_customers_dataset.csv
data\olist_geolocation_dataset.csv
data\olist_order_items_dataset.csv
data\olist_order_payments_dataset.csv
data\olist_order_reviews_dataset.csv
data\olist_orders_dataset.csv
data\olist_products_dataset.csv
data\olist_sellers_dataset.csv
data\processed\customer_features.parquet
data\processed\customer_features.pkl
data\processed\customer_intelligence_historical.csv
data\processed\customer_intelligence_historical.parquet
data\processed\dataset_inventory.csv
data\product_category_name_translation.csv


In [2]:
# Load the original Olist transaction tables
customers = pd.read_csv(data_directory / "olist_customers_dataset.csv")
orders = pd.read_csv(data_directory / "olist_orders_dataset.csv")
order_items = pd.read_csv(data_directory / "olist_order_items_dataset.csv")
payments = pd.read_csv(data_directory / "olist_order_payments_dataset.csv")

# Inspect table structure, identifying columns and missing values
source_tables = {
    "Customers": customers,
    "Orders": orders,
    "Order Items": order_items,
    "Payments": payments,
}

for table_name, df in source_tables.items():
    print(f"\n{'=' * 55}")
    print(f"{table_name}: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"{'=' * 55}")

    display(pd.DataFrame({
        "Data Type": df.dtypes.astype(str),
        "Missing Values": df.isna().sum(),
        "Unique Values": df.nunique(dropna=True)
    }))


Customers: 99,441 rows × 5 columns


,Data Type,Missing Values,Unique Values
customer_id,str,0,99441
customer_unique_id,str,0,96096
customer_zip_code_prefix,int64,0,14994
customer_city,str,0,4119
customer_state,str,0,27



Orders: 99,441 rows × 8 columns


,Data Type,Missing Values,Unique Values
order_id,str,0,99441
customer_id,str,0,99441
order_status,str,0,8
order_purchase_timestamp,str,0,98875
order_approved_at,str,160,90733
order_delivered_carrier_date,str,1783,81018
order_delivered_customer_date,str,2965,95664
order_estimated_delivery_date,str,0,459



Order Items: 112,650 rows × 7 columns


,Data Type,Missing Values,Unique Values
order_id,str,0,98666
order_item_id,int64,0,21
product_id,str,0,32951
seller_id,str,0,3095
shipping_limit_date,str,0,93318
price,float64,0,5968
freight_value,float64,0,6999



Payments: 103,886 rows × 5 columns


,Data Type,Missing Values,Unique Values
order_id,str,0,99440
payment_sequential,int64,0,29
payment_type,str,0,5
payment_installments,int64,0,24
payment_value,float64,0,29077


In [3]:
# Inspect order status and financial record coverage
print("ORDER STATUS DISTRIBUTION")
display(
    orders["order_status"]
    .value_counts(dropna=False)
    .rename_axis("order_status")
    .reset_index(name="order_count")
)

# Aggregate item and payment values separately to order level
item_totals = (
    order_items
    .assign(item_total=lambda df: df["price"] + df["freight_value"])
    .groupby("order_id", as_index=False)
    .agg(
        product_revenue=("price", "sum"),
        freight_revenue=("freight_value", "sum"),
        item_order_total=("item_total", "sum")
    )
)

payment_totals = (
    payments
    .groupby("order_id", as_index=False)
    .agg(payment_order_total=("payment_value", "sum"))
)

# Use one record per order to prevent duplicated revenue
order_audit = (
    orders[
        ["order_id", "order_status", "order_purchase_timestamp"]
    ]
    .merge(
        item_totals,
        on="order_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        payment_totals,
        on="order_id",
        how="left",
        validate="one_to_one"
    )
)

order_audit["payment_difference"] = (
    order_audit["payment_order_total"]
    - order_audit["item_order_total"]
)

print("\nFINANCIAL COVERAGE")
print(f"Total orders: {len(order_audit):,}")
print(f"Orders with item records: {order_audit['item_order_total'].notna().sum():,}")
print(f"Orders with payment records: {order_audit['payment_order_total'].notna().sum():,}")
print(f"Orders with both: {order_audit[['item_order_total', 'payment_order_total']].notna().all(axis=1).sum():,}")

print("\nPAYMENT RECONCILIATION")
matched_orders = order_audit.dropna(
    subset=["item_order_total", "payment_order_total"]
)

print(
    "Orders with payment and item totals differing by more than R$0.01:",
    (matched_orders["payment_difference"].abs() > 0.01).sum()
)

display(
    matched_orders["payment_difference"].describe()
)

ORDER STATUS DISTRIBUTION


,order_status,order_count
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2



FINANCIAL COVERAGE
Total orders: 99,441
Orders with item records: 98,666
Orders with payment records: 99,440
Orders with both: 98,665

PAYMENT RECONCILIATION
Orders with payment and item totals differing by more than R$0.01: 378


count    98665.000000
mean         0.029092
std          1.129221
min        -51.620000
25%          0.000000
50%          0.000000
75%          0.000000
max        182.810000
Name: payment_difference, dtype: float64

In [4]:
# Examine financial reconciliation by order status
# Keep unmatched orders visible rather than treating missing values as zero

order_audit["financial_record_status"] = np.select(
    [
        order_audit["item_order_total"].notna()
        & order_audit["payment_order_total"].notna(),

        order_audit["item_order_total"].isna()
        & order_audit["payment_order_total"].notna(),

        order_audit["item_order_total"].notna()
        & order_audit["payment_order_total"].isna()
    ],
    [
        "Both records",
        "Payment only",
        "Items only"
    ],
    default="Neither"
)

order_audit["financial_discrepancy"] = (
    order_audit["item_order_total"].notna()
    & order_audit["payment_order_total"].notna()
    & (order_audit["payment_difference"].abs() > 0.01)
)

status_audit = (
    order_audit
    .groupby("order_status", as_index=False)
    .agg(
        total_orders=("order_id", "count"),
        matched_financial_records=(
            "financial_record_status",
            lambda s: (s == "Both records").sum()
        ),
        missing_item_records=(
            "item_order_total",
            lambda s: s.isna().sum()
        ),
        missing_payment_records=(
            "payment_order_total",
            lambda s: s.isna().sum()
        ),
        discrepant_orders=("financial_discrepancy", "sum")
    )
)

status_audit["discrepancy_rate_pct"] = (
    100 * status_audit["discrepant_orders"]
    / status_audit["matched_financial_records"].replace(0, np.nan)
)

display(
    status_audit.sort_values(
        "total_orders", ascending=False
    ).round(2)
)

print("\nLARGEST FINANCIAL DISCREPANCIES")

display(
    order_audit.loc[
        order_audit["financial_discrepancy"],
        [
            "order_id",
            "order_status",
            "item_order_total",
            "payment_order_total",
            "payment_difference"
        ]
    ]
    .assign(
        absolute_difference=lambda df:
            df["payment_difference"].abs()
    )
    .sort_values("absolute_difference", ascending=False)
    .head(10)
)

,order_status,total_orders,matched_financial_records,missing_item_records,missing_payment_records,discrepant_orders,discrepancy_rate_pct
3,delivered,96478,96477,0,1,372,0.39
6,shipped,1107,1106,1,0,3,0.27
1,canceled,625,461,164,0,2,0.43
7,unavailable,609,6,603,0,0,0.00
4,invoiced,314,312,2,0,1,0.32
5,processing,301,301,0,0,0,0.00
2,created,5,0,5,0,0,NaN
0,approved,2,2,0,0,0,0.00



LARGEST FINANCIAL DISCREPANCIES


,order_id,order_status,item_order_total,payment_order_total,payment_difference,absolute_difference
11791,ce6d150fb29ada17d2082f4847107665,delivered,1403.66,1586.47,182.81,182.81
48686,6e5fe7366a2e1bfbf3257dba0af1267f,delivered,287.91,406.92,119.01,119.01
70865,70b742795bc441e94a44a084b6d9ce7a,delivered,466.93,578.82,111.89,111.89
33150,996c7e73600ad3723e8627ab7bef81e4,delivered,587.90,664.43,76.53,76.53
52985,70b7e94ea46d3e8b5bc12a50186edaf0,delivered,213.15,274.84,61.69,61.69
85434,bc2c82b0ef78d2252b6176d1972db7c9,delivered,242.01,303.02,61.01,61.01
94304,af9ffff2ce6b3defd34fd4c78857a379,delivered,413.17,466.97,53.80,53.80
31661,262118ce178bb3e4590a3adcf6d62e6b,delivered,177.74,126.12,-51.62,51.62
68811,bfdb5bbb06458d600a33d61f5f287472,delivered,348.93,394.36,45.43,45.43
8548,8d9c0dc8d5a2ce804f6b925d8f8e6c1d,delivered,254.45,293.89,39.44,39.44


In [5]:
# Quantify financial discrepancies for delivered orders only

delivered_orders = order_audit.loc[
    order_audit["order_status"] == "delivered"
].copy()

delivered_matched = delivered_orders.dropna(
    subset=["item_order_total", "payment_order_total"]
).copy()

delivered_discrepancies = delivered_matched.loc[
    delivered_matched["payment_difference"].abs() > 0.01
]

revenue_audit = pd.DataFrame({
    "Metric": [
        "Total delivered orders",
        "Delivered orders with both financial records",
        "Delivered orders with financial discrepancies",
        "Percentage of matched delivered orders with discrepancies",
        "Total item value including freight (matched delivered orders)",
        "Total payment value (matched delivered orders)",
        "Net payment minus item value",
        "Total absolute financial difference"
    ],
    "Value": [
        len(delivered_orders),
        len(delivered_matched),
        len(delivered_discrepancies),
        100 * len(delivered_discrepancies) / len(delivered_matched),
        delivered_matched["item_order_total"].sum(),
        delivered_matched["payment_order_total"].sum(),
        delivered_matched["payment_difference"].sum(),
        delivered_matched["payment_difference"].abs().sum()
    ]
})

display(revenue_audit)

print("\nDELIVERED ORDER FINANCIAL RECORD COVERAGE")

display(
    delivered_orders["financial_record_status"]
    .value_counts()
    .rename_axis("financial_record_status")
    .reset_index(name="order_count")
)

,Metric,Value
0,Total delivered orders,9.647800e+04
1,Delivered orders with both financial records,9.647700e+04
2,Delivered orders with financial discrepancies,3.720000e+02
3,Percentage of matched delivered orders with di...,3.855841e-01
4,Total item value including freight (matched de...,1.541963e+07
5,Total payment value (matched delivered orders),1.542246e+07
6,Net payment minus item value,2.831480e+03
7,Total absolute financial difference,3.233000e+03



DELIVERED ORDER FINANCIAL RECORD COVERAGE


,financial_record_status,order_count
0,Both records,96477
1,Items only,1


In [6]:
# Link delivered orders to persistent customer identifiers
delivered_customer_orders = (
    orders.loc[
        orders["order_status"] == "delivered",
        ["order_id", "customer_id", "order_purchase_timestamp"]
    ]
    .merge(
        customers[["customer_id", "customer_unique_id"]],
        on="customer_id",
        how="left",
        validate="many_to_one"
    )
)

# Load the existing customer intelligence population
historical_intelligence = pd.read_parquet(
    data_directory / "processed" / "customer_intelligence_historical.parquet"
)

# Compare customer populations without changing either dataset
delivered_customer_ids = set(
    delivered_customer_orders["customer_unique_id"].dropna()
)

intelligence_customer_ids = set(
    historical_intelligence["customer_unique_id"].dropna()
)

population_comparison = pd.DataFrame({
    "Population": [
        "Unique customers with delivered orders",
        "Customers in existing intelligence platform",
        "Customers appearing in both populations",
        "Delivered-order customers outside intelligence platform",
        "Intelligence customers without delivered orders"
    ],
    "Customer Count": [
        len(delivered_customer_ids),
        len(intelligence_customer_ids),
        len(delivered_customer_ids & intelligence_customer_ids),
        len(delivered_customer_ids - intelligence_customer_ids),
        len(intelligence_customer_ids - delivered_customer_ids)
    ]
})

display(population_comparison)

print(
    "\nMissing customer identifiers:",
    delivered_customer_orders["customer_unique_id"].isna().sum()
)

print(
    "Duplicate order identifiers:",
    delivered_customer_orders["order_id"].duplicated().sum()
)

,Population,Customer Count
0,Unique customers with delivered orders,93358
1,Customers in existing intelligence platform,78100
2,Customers appearing in both populations,75699
3,Delivered-order customers outside intelligence...,17659
4,Intelligence customers without delivered orders,2401



Missing customer identifiers: 0
Duplicate order identifiers: 0


In [7]:
# Establish the historical observation window for delivered orders

delivered_customer_orders["order_purchase_timestamp"] = pd.to_datetime(
    delivered_customer_orders["order_purchase_timestamp"],
    errors="coerce"
)

observation_cutoff = pd.Timestamp("2018-08-31 23:59:59")

observation_audit = pd.DataFrame({
    "Metric": [
        "Earliest delivered purchase",
        "Latest delivered purchase",
        "Delivered orders with invalid purchase timestamps",
        "Delivered orders on or before platform cutoff",
        "Delivered orders after platform cutoff",
        "Unique customers on or before platform cutoff",
        "Unique customers after platform cutoff"
    ],
    "Value": [
        delivered_customer_orders["order_purchase_timestamp"].min(),
        delivered_customer_orders["order_purchase_timestamp"].max(),
        delivered_customer_orders["order_purchase_timestamp"].isna().sum(),
        (
            delivered_customer_orders["order_purchase_timestamp"]
            <= observation_cutoff
        ).sum(),
        (
            delivered_customer_orders["order_purchase_timestamp"]
            > observation_cutoff
        ).sum(),
        delivered_customer_orders.loc[
            delivered_customer_orders["order_purchase_timestamp"]
            <= observation_cutoff,
            "customer_unique_id"
        ].nunique(),
        delivered_customer_orders.loc[
            delivered_customer_orders["order_purchase_timestamp"]
            > observation_cutoff,
            "customer_unique_id"
        ].nunique()
    ]
})

display(observation_audit)

,Metric,Value
0,Earliest delivered purchase,2016-09-15 12:16:38
1,Latest delivered purchase,2018-08-29 15:00:37
2,Delivered orders with invalid purchase timestamps,0
3,Delivered orders on or before platform cutoff,96478
4,Delivered orders after platform cutoff,0
5,Unique customers on or before platform cutoff,93358
6,Unique customers after platform cutoff,0


In [8]:
# Inspect the first-purchase observation window
# for customers already included in our intelligence platform

historical_intelligence["order_purchase_timestamp"] = pd.to_datetime(
    historical_intelligence["order_purchase_timestamp"],
    errors="coerce"
)

first_purchase_audit = pd.DataFrame({
    "Metric": [
        "Customers in existing intelligence platform",
        "Earliest first purchase",
        "Latest first purchase",
        "Missing first purchase timestamps",
        "Customers with first purchase after 2 June 2018",
        "Customers with complete 60-day observation plus 30-day buffer"
    ],
    "Value": [
        len(historical_intelligence),
        historical_intelligence["order_purchase_timestamp"].min(),
        historical_intelligence["order_purchase_timestamp"].max(),
        historical_intelligence["order_purchase_timestamp"].isna().sum(),
        (
            historical_intelligence["order_purchase_timestamp"]
            > pd.Timestamp("2018-06-02 23:59:59")
        ).sum(),
        (
            historical_intelligence["order_purchase_timestamp"]
            + pd.Timedelta(days=90)
            <= observation_cutoff
        ).sum()
    ]
})

display(first_purchase_audit)

,Metric,Value
0,Customers in existing intelligence platform,78100
1,Earliest first purchase,2016-09-04 21:15:19
2,Latest first purchase,2018-06-02 23:42:33
3,Missing first purchase timestamps,0
4,Customers with first purchase after 2 June 2018,0
5,Customers with complete 60-day observation plu...,78100


In [9]:
# Validate the original first-order population against order statuses
# Preserve the first-order definition used in earlier notebooks

first_order_audit = (
    historical_intelligence[
        [
            "customer_unique_id",
            "order_id",
            "order_purchase_timestamp"
        ]
    ]
    .merge(
        orders[["order_id", "order_status"]],
        on="order_id",
        how="left",
        validate="one_to_one"
    )
)

print("FIRST-ORDER STATUS DISTRIBUTION")

display(
    first_order_audit["order_status"]
    .value_counts(dropna=False)
    .rename_axis("first_order_status")
    .reset_index(name="customer_count")
)

print("\nFIRST-ORDER VALIDATION")

print(
    "Customers:",
    f"{len(first_order_audit):,}"
)

print(
    "Missing order statuses:",
    first_order_audit["order_status"].isna().sum()
)

print(
    "First orders marked as delivered:",
    (first_order_audit["order_status"] == "delivered").sum()
)

print(
    "First orders not marked as delivered:",
    (
        first_order_audit["order_status"].notna()
        & (first_order_audit["order_status"] != "delivered")
    ).sum()
)

FIRST-ORDER STATUS DISTRIBUTION


,first_order_status,customer_count
0,delivered,75599
1,shipped,928
2,unavailable,560
3,canceled,444
4,processing,293
5,invoiced,269
6,created,5
7,approved,2



FIRST-ORDER VALIDATION
Customers: 78,100
Missing order statuses: 0
First orders marked as delivered: 75599
First orders not marked as delivered: 2501


In [10]:
# Build a delivered-order merchandise value foundation
# Retain product value separately from freight and payment totals

delivered_order_value = (
    delivered_customer_orders
    .merge(
        item_totals[
            [
                "order_id",
                "product_revenue",
                "freight_revenue",
                "item_order_total"
            ]
        ],
        on="order_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        payment_totals[
            ["order_id", "payment_order_total"]
        ],
        on="order_id",
        how="left",
        validate="one_to_one"
    )
)

# Validate that joins preserve the delivered-order population
assert len(delivered_order_value) == len(delivered_customer_orders)

assert delivered_order_value["order_id"].is_unique

assert delivered_order_value["customer_unique_id"].notna().all()

print("DELIVERED ORDER VALUE FOUNDATION")

print(f"Delivered orders: {len(delivered_order_value):,}")

print(
    "Unique customers:",
    f"{delivered_order_value['customer_unique_id'].nunique():,}"
)

print(
    "Missing merchandise values:",
    delivered_order_value["product_revenue"].isna().sum()
)

print(
    "Missing payment values:",
    delivered_order_value["payment_order_total"].isna().sum()
)

print(
    "Total observed merchandise value:",
    f"R${delivered_order_value['product_revenue'].sum():,.2f}"
)

print(
    "Total observed freight value:",
    f"R${delivered_order_value['freight_revenue'].sum():,.2f}"
)

display(delivered_order_value.head())

DELIVERED ORDER VALUE FOUNDATION
Delivered orders: 96,478
Unique customers: 93,358
Missing merchandise values: 0
Missing payment values: 1
Total observed merchandise value: R$13,221,498.11
Total observed freight value: R$2,198,275.64


,order_id,customer_id,order_purchase_timestamp,customer_unique_id,product_revenue,freight_revenue,item_order_total,payment_order_total
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,2017-10-02 10:56:33,7c396fd4830fd04220f754e42b4e5bff,29.99,8.72,38.71,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,2018-07-24 20:41:37,af07308b275d755c9edb36a90c618231,118.70,22.76,141.46,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,2018-08-08 08:38:49,3a653a41f6f9fc3d2a113cf8398680e8,159.90,19.22,179.12,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,2017-11-18 19:28:06,7c142cf63193a1473d2e66489a9ae977,45.00,27.20,72.20,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,2018-02-13 21:18:39,72632f0f9dd73dfee390c9b22eb56dd6,19.90,8.72,28.62,28.62


In [11]:
# Confirm the complete delivered-order value dataset is valid

validation_summary = pd.DataFrame({
    "Metric": [
        "Delivered orders",
        "Unique customers",
        "Duplicate order identifiers",
        "Missing customer identifiers",
        "Missing merchandise values",
        "Missing payment values",
        "Total merchandise value (R$)",
        "Total freight value (R$)"
    ],
    "Value": [
        len(delivered_order_value),
        delivered_order_value["customer_unique_id"].nunique(),
        delivered_order_value["order_id"].duplicated().sum(),
        delivered_order_value["customer_unique_id"].isna().sum(),
        delivered_order_value["product_revenue"].isna().sum(),
        delivered_order_value["payment_order_total"].isna().sum(),
        round(delivered_order_value["product_revenue"].sum(), 2),
        round(delivered_order_value["freight_revenue"].sum(), 2)
    ]
})

display(validation_summary)

,Metric,Value
0,Delivered orders,96478.00
1,Unique customers,93358.00
2,Duplicate order identifiers,0.00
3,Missing customer identifiers,0.00
4,Missing merchandise values,0.00
5,Missing payment values,1.00
6,Total merchandise value (R$),13221498.11
7,Total freight value (R$),2198275.64


In [12]:
# Build observed 60-day customer merchandise value
# Use each customer's original first recorded purchase as the starting point

customer_first_orders = historical_intelligence[
    [
        "customer_unique_id",
        "order_id",
        "order_purchase_timestamp"
    ]
].rename(
    columns={
        "order_id": "first_order_id",
        "order_purchase_timestamp": "first_purchase_timestamp"
    }
)

# Link each customer to their delivered orders
customer_value_orders = (
    customer_first_orders
    .merge(
        delivered_order_value[
            [
                "customer_unique_id",
                "order_id",
                "order_purchase_timestamp",
                "product_revenue"
            ]
        ],
        on="customer_unique_id",
        how="left",
        validate="one_to_many",
        suffixes=("", "_delivered")
    )
)

# Calculate elapsed time since the original first purchase
customer_value_orders["days_since_first_purchase"] = (
    customer_value_orders["order_purchase_timestamp"]
    - customer_value_orders["first_purchase_timestamp"]
).dt.total_seconds() / 86400

# Include only delivered orders within the first 60 days
customer_value_orders["within_60_days"] = (
    customer_value_orders["days_since_first_purchase"]
    .between(0, 60, inclusive="both")
)

# Distinguish first-order value from subsequent purchasing value
customer_value_orders["initial_delivered_value"] = np.where(
    customer_value_orders["within_60_days"]
    & (
        customer_value_orders["order_id"]
        == customer_value_orders["first_order_id"]
    ),
    customer_value_orders["product_revenue"],
    0.0
)

customer_value_orders["subsequent_60d_value"] = np.where(
    customer_value_orders["within_60_days"]
    & (
        customer_value_orders["order_id"]
        != customer_value_orders["first_order_id"]
    ),
    customer_value_orders["product_revenue"],
    0.0
)

# Aggregate back to exactly one row per existing customer
customer_value_60d = (
    customer_value_orders
    .groupby("customer_unique_id", as_index=False)
    .agg(
        initial_delivered_value=("initial_delivered_value", "sum"),
        subsequent_60d_value=("subsequent_60d_value", "sum")
    )
)

customer_value_60d["total_60d_merchandise_value"] = (
    customer_value_60d["initial_delivered_value"]
    + customer_value_60d["subsequent_60d_value"]
)

# Validate population and financial consistency
assert len(customer_value_60d) == 78_100
assert customer_value_60d["customer_unique_id"].is_unique
assert (
    customer_value_60d["total_60d_merchandise_value"]
    >= 0
).all()

assert np.allclose(
    customer_value_60d["total_60d_merchandise_value"],
    customer_value_60d["initial_delivered_value"]
    + customer_value_60d["subsequent_60d_value"]
)

print("CUSTOMER VALUE DATASET CREATED")
print(f"Customers: {len(customer_value_60d):,}")

print("\nHISTORICAL 60-DAY MERCHANDISE VALUE SUMMARY")

display(
    customer_value_60d[
        [
            "initial_delivered_value",
            "subsequent_60d_value",
            "total_60d_merchandise_value"
        ]
    ].describe().round(2)
)

print("\nCUSTOMERS GENERATING SUBSEQUENT VALUE")

print(
    "Customers with positive subsequent merchandise value:",
    f"{(customer_value_60d['subsequent_60d_value'] > 0).sum():,}"
)

print(
    "Customers with zero delivered merchandise value:",
    f"{(customer_value_60d['total_60d_merchandise_value'] == 0).sum():,}"
)

CUSTOMER VALUE DATASET CREATED
Customers: 78,100

HISTORICAL 60-DAY MERCHANDISE VALUE SUMMARY


,initial_delivered_value,subsequent_60d_value,total_60d_merchandise_value
count,78100.00,78100.00,78100.00
mean,133.10,2.71,135.81
std,206.92,31.53,210.58
min,0.00,0.00,0.00
25%,42.00,0.00,43.01
50%,82.00,0.00,84.99
75%,149.51,0.00,149.90
max,13440.00,2589.00,13440.00



CUSTOMERS GENERATING SUBSEQUENT VALUE
Customers with positive subsequent merchandise value: 1,589
Customers with zero delivered merchandise value: 2,432


In [13]:
# Reconcile repeat-purchase labels with subsequent delivered merchandise value

value_reconciliation = (
    historical_intelligence[
        ["customer_unique_id", "repeat_purchase_60d"]
    ]
    .merge(
        customer_value_60d,
        on="customer_unique_id",
        how="left",
        validate="one_to_one"
    )
)

value_reconciliation["has_subsequent_delivered_value"] = (
    value_reconciliation["subsequent_60d_value"] > 0
)

print("REPEAT PURCHASE AND SUBSEQUENT VALUE RECONCILIATION")

display(
    pd.crosstab(
        value_reconciliation["repeat_purchase_60d"],
        value_reconciliation["has_subsequent_delivered_value"],
        rownames=["Repeat purchase label"],
        colnames=["Generated subsequent delivered value"],
        margins=True
    )
)

print("\nCUSTOMER-LEVEL DISAGREEMENTS")

disagreements = value_reconciliation.loc[
    value_reconciliation["repeat_purchase_60d"].astype(bool)
    != value_reconciliation["has_subsequent_delivered_value"]
]

print(f"Customers with differing classifications: {len(disagreements):,}")

display(
    disagreements[
        [
            "customer_unique_id",
            "repeat_purchase_60d",
            "initial_delivered_value",
            "subsequent_60d_value"
        ]
    ].head(10)
)

REPEAT PURCHASE AND SUBSEQUENT VALUE RECONCILIATION


Generated subsequent delivered value,False,True,All
Repeat purchase label,,,
0,76511,0,76511
1,0,1589,1589
All,76511,1589,78100



CUSTOMER-LEVEL DISAGREEMENTS
Customers with differing classifications: 0


,customer_unique_id,repeat_purchase_60d,initial_delivered_value,subsequent_60d_value


In [14]:
# Establish the historical 60-day customer-value baseline
# Report value separately from repeat-purchase incidence

value_columns = [
    "initial_delivered_value",
    "subsequent_60d_value",
    "total_60d_merchandise_value"
]

value_baseline = pd.DataFrame({
    "Metric": value_columns,
    "Total Value (R$)": [
        customer_value_60d[col].sum()
        for col in value_columns
    ],
    "Average per Customer (R$)": [
        customer_value_60d[col].mean()
        for col in value_columns
    ],
    "Median per Customer (R$)": [
        customer_value_60d[col].median()
        for col in value_columns
    ],
    "Customers with Positive Value": [
        customer_value_60d[col].gt(0).sum()
        for col in value_columns
    ]
})

display(value_baseline.round(2))

# Examine subsequent spending among customers who actually repurchased
repeat_value_distribution = (
    customer_value_60d.loc[
        customer_value_60d["subsequent_60d_value"] > 0,
        "subsequent_60d_value"
    ]
)

print("\nSUBSEQUENT VALUE AMONG REPEAT PURCHASERS")

display(
    repeat_value_distribution.describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    ).round(2)
)

print("\nSHARE OF TOTAL 60-DAY VALUE FROM SUBSEQUENT PURCHASES")

total_value = customer_value_60d[
    "total_60d_merchandise_value"
].sum()

subsequent_share = (
    customer_value_60d["subsequent_60d_value"].sum()
    / total_value * 100
)

print(f"{subsequent_share:.2f}%")

,Metric,Total Value (R$),Average per Customer (R$),Median per Customer (R$),Customers with Positive Value
0,initial_delivered_value,10395105.68,133.10,82.00,75599
1,subsequent_60d_value,211295.19,2.71,0.00,1589
2,total_60d_merchandise_value,10606400.87,135.81,84.99,75668



SUBSEQUENT VALUE AMONG REPEAT PURCHASERS


count    1589.00
mean      132.97
std       177.65
min         5.99
25%        45.90
50%        84.89
75%       154.00
90%       260.40
95%       397.56
99%       949.91
max      2589.00
Name: subsequent_60d_value, dtype: float64


SHARE OF TOTAL 60-DAY VALUE FROM SUBSEQUENT PURCHASES
1.99%


In [15]:
display(value_baseline.round(2))

,Metric,Total Value (R$),Average per Customer (R$),Median per Customer (R$),Customers with Positive Value
0,initial_delivered_value,10395105.68,133.10,82.00,75599
1,subsequent_60d_value,211295.19,2.71,0.00,1589
2,total_60d_merchandise_value,10606400.87,135.81,84.99,75668


In [16]:
# Investigate the relationship between initial purchase value
# and additional merchandise value generated within 60 days

customer_value_analysis = (
    historical_intelligence[
        [
            "customer_unique_id",
            "first_order_item_value",
            "repeat_purchase_propensity",
            "propensity_segment",
            "initial_value_segment",
            "customer_profile"
        ]
    ]
    .merge(
        customer_value_60d,
        on="customer_unique_id",
        how="left",
        validate="one_to_one"
    )
)

# Preserve the original first-order feature for comparison
# and use validated delivered merchandise value as the revenue measure
customer_value_analysis["generated_subsequent_value"] = (
    customer_value_analysis["subsequent_60d_value"] > 0
)

# Create value bands using customers with positive initial delivered value
positive_initial_value = (
    customer_value_analysis["initial_delivered_value"] > 0
)

customer_value_analysis["initial_value_band"] = "No delivered initial value"

customer_value_analysis.loc[
    positive_initial_value, "initial_value_band"
] = pd.qcut(
    customer_value_analysis.loc[
        positive_initial_value, "initial_delivered_value"
    ],
    q=4,
    labels=[
        "Q1: Lowest",
        "Q2: Lower-middle",
        "Q3: Upper-middle",
        "Q4: Highest"
    ],
    duplicates="drop"
).astype(str)

# Summarise repeat behaviour and subsequent value by initial-value band
initial_value_summary = (
    customer_value_analysis
    .groupby("initial_value_band", as_index=False)
    .agg(
        customers=("customer_unique_id", "count"),
        repeat_customers=("generated_subsequent_value", "sum"),
        average_initial_value=("initial_delivered_value", "mean"),
        average_subsequent_value=("subsequent_60d_value", "mean"),
        total_subsequent_value=("subsequent_60d_value", "sum")
    )
)

initial_value_summary["repeat_rate_pct"] = (
    100
    * initial_value_summary["repeat_customers"]
    / initial_value_summary["customers"]
)

display(initial_value_summary.round(2))

,initial_value_band,customers,repeat_customers,average_initial_value,average_subsequent_value,total_subsequent_value,repeat_rate_pct
0,No delivered initial value,2501,69,0.00,5.39,13477.05,2.76
1,Q1: Lowest,18922,414,28.59,1.78,33736.14,2.19
2,Q2: Lower-middle,18907,409,63.90,2.18,41272.46,2.16
3,Q3: Upper-middle,18922,365,114.93,2.38,44973.86,1.93
4,Q4: Highest,18848,332,343.34,4.13,77835.68,1.76


In [17]:
# Evaluate observed subsequent merchandise value by existing customer profile
# Distinguish repeat-purchase incidence from actual subsequent spending

profile_value_summary = (
    customer_value_analysis
    .groupby("customer_profile", dropna=False)
    .agg(
        customers=("customer_unique_id", "count"),
        repeat_customers=("generated_subsequent_value", "sum"),
        total_initial_value=("initial_delivered_value", "sum"),
        total_subsequent_value=("subsequent_60d_value", "sum"),
        average_initial_value=("initial_delivered_value", "mean"),
        average_subsequent_value=("subsequent_60d_value", "mean"),
        average_total_60d_value=("total_60d_merchandise_value", "mean")
    )
    .reset_index()
)

profile_value_summary["repeat_rate_pct"] = (
    100
    * profile_value_summary["repeat_customers"]
    / profile_value_summary["customers"]
)

profile_value_summary["subsequent_value_per_repeater"] = (
    profile_value_summary["total_subsequent_value"]
    / profile_value_summary["repeat_customers"].replace(0, np.nan)
)

# Validate that all customers and subsequent value are accounted for
assert profile_value_summary["customers"].sum() == 78_100

assert np.isclose(
    profile_value_summary["total_subsequent_value"].sum(),
    customer_value_analysis["subsequent_60d_value"].sum()
)

display(
    profile_value_summary.sort_values(
        "average_subsequent_value",
        ascending=False
    ).round(2)
)

,customer_profile,customers,repeat_customers,total_initial_value,total_subsequent_value,average_initial_value,average_subsequent_value,average_total_60d_value,repeat_rate_pct,subsequent_value_per_repeater
6,Unknown Initial Value,642,21,0.00,4062.43,0.00,6.33,6.33,3.27,193.45
0,Higher Value | High Propensity,7287,259,1239690.93,40631.07,170.12,5.58,175.70,3.55,156.88
2,Higher Value | Medium Propensity,10543,209,1932099.74,31855.87,183.26,3.02,186.28,1.98,152.42
3,Lower Value | High Propensity,7814,284,388633.46,22825.82,49.74,2.92,52.66,3.63,80.37
1,Higher Value | Low Propensity,20931,249,5474097.04,57619.49,261.53,2.75,264.28,1.19,231.40
5,Lower Value | Medium Propensity,12766,282,574069.31,26342.29,44.97,2.06,47.03,2.21,93.41
4,Lower Value | Low Propensity,18117,285,786515.20,27958.22,43.41,1.54,44.96,1.57,98.10


In [18]:
# Decompose subsequent customer value into two components:
# repeat-purchase frequency and spending among repeat purchasers

value_decomposition = profile_value_summary[
    [
        "customer_profile",
        "customers",
        "repeat_customers",
        "repeat_rate_pct",
        "subsequent_value_per_repeater",
        "average_subsequent_value"
    ]
].copy()

# Reconstruct expected historical value per customer
# using observed repeat incidence and spending intensity
value_decomposition["reconstructed_value"] = (
    value_decomposition["repeat_rate_pct"] / 100
    * value_decomposition["subsequent_value_per_repeater"]
)

# Check that the decomposition reconciles
assert np.allclose(
    value_decomposition["average_subsequent_value"],
    value_decomposition["reconstructed_value"],
    rtol=1e-8,
    atol=1e-8
)

display(
    value_decomposition.sort_values(
        "average_subsequent_value",
        ascending=False
    ).round(2)
)

,customer_profile,customers,repeat_customers,repeat_rate_pct,subsequent_value_per_repeater,average_subsequent_value,reconstructed_value
6,Unknown Initial Value,642,21,3.27,193.45,6.33,6.33
0,Higher Value | High Propensity,7287,259,3.55,156.88,5.58,5.58
2,Higher Value | Medium Propensity,10543,209,1.98,152.42,3.02,3.02
3,Lower Value | High Propensity,7814,284,3.63,80.37,2.92,2.92
1,Higher Value | Low Propensity,20931,249,1.19,231.40,2.75,2.75
5,Lower Value | Medium Propensity,12766,282,2.21,93.41,2.06,2.06
4,Lower Value | Low Propensity,18117,285,1.57,98.10,1.54,1.54


In [19]:
# Assess customer eligibility for longer historical value horizons
# Preserve the original observation cutoff and first-purchase dates

value_horizons = [60, 90, 180, 365]

horizon_eligibility = []

for horizon_days in value_horizons:
    # Include the established 30-day delivery buffer
    required_days = horizon_days + 30

    eligible = (
        historical_intelligence["order_purchase_timestamp"]
        + pd.Timedelta(days=required_days)
        <= observation_cutoff
    )

    horizon_eligibility.append({
        "Value Horizon (Days)": horizon_days,
        "Required Observation (Days)": required_days,
        "Eligible Customers": int(eligible.sum()),
        "Ineligible Customers": int((~eligible).sum()),
        "Eligible Share (%)": round(eligible.mean() * 100, 2)
    })

horizon_eligibility_summary = pd.DataFrame(horizon_eligibility)

display(horizon_eligibility_summary)

,Value Horizon (Days),Required Observation (Days),Eligible Customers,Ineligible Customers,Eligible Share (%)
0,60,90,78100,0,100.00
1,90,120,71988,6112,92.17
2,180,210,51461,26639,65.89
3,365,395,18606,59494,23.82


In [20]:
# Compare observed 60-day and 90-day merchandise value
# using only customers eligible for the full 90-day observation window

eligible_90d_ids = set(
    historical_intelligence.loc[
        historical_intelligence["order_purchase_timestamp"]
        + pd.Timedelta(days=120)
        <= observation_cutoff,
        "customer_unique_id"
    ]
)

# Retain the exact same customer population for both horizons
eligible_value_orders = customer_value_orders.loc[
    customer_value_orders["customer_unique_id"].isin(eligible_90d_ids)
].copy()

# Measure delivered merchandise value by purchase timing
eligible_value_orders["value_60d"] = np.where(
    eligible_value_orders["days_since_first_purchase"].between(
        0, 60, inclusive="both"
    ),
    eligible_value_orders["product_revenue"],
    0.0
)

eligible_value_orders["value_90d"] = np.where(
    eligible_value_orders["days_since_first_purchase"].between(
        0, 90, inclusive="both"
    ),
    eligible_value_orders["product_revenue"],
    0.0
)

eligible_value_orders["incremental_value_days_61_to_90"] = (
    eligible_value_orders["value_90d"]
    - eligible_value_orders["value_60d"]
)

# Aggregate to customer level
customer_value_90d = (
    eligible_value_orders
    .groupby("customer_unique_id", as_index=False)
    .agg(
        value_60d=("value_60d", "sum"),
        value_90d=("value_90d", "sum"),
        incremental_value_days_61_to_90=(
            "incremental_value_days_61_to_90", "sum"
        )
    )
)

# Validate the population and value reconciliation
assert len(customer_value_90d) == 71_988
assert customer_value_90d["customer_unique_id"].is_unique

assert np.allclose(
    customer_value_90d["value_90d"],
    customer_value_90d["value_60d"]
    + customer_value_90d["incremental_value_days_61_to_90"]
)

horizon_comparison = pd.DataFrame({
    "Metric": [
        "Eligible customers",
        "Total 60-day merchandise value (R$)",
        "Total 90-day merchandise value (R$)",
        "Additional merchandise value from days 61–90 (R$)",
        "Average 60-day value per customer (R$)",
        "Average 90-day value per customer (R$)",
        "Customers generating additional value during days 61–90"
    ],
    "Value": [
        len(customer_value_90d),
        customer_value_90d["value_60d"].sum(),
        customer_value_90d["value_90d"].sum(),
        customer_value_90d["incremental_value_days_61_to_90"].sum(),
        customer_value_90d["value_60d"].mean(),
        customer_value_90d["value_90d"].mean(),
        (
            customer_value_90d["incremental_value_days_61_to_90"] > 0
        ).sum()
    ]
})

display(horizon_comparison.round(2))

,Metric,Value
0,Eligible customers,71988.00
1,Total 60-day merchandise value (R$),9698188.11
2,Total 90-day merchandise value (R$),9726004.75
3,Additional merchandise value from days 61–90 (R$),27816.64
4,Average 60-day value per customer (R$),134.72
5,Average 90-day value per customer (R$),135.11
6,Customers generating additional value during d...,200.00


In [21]:
# Distinguish new repeat purchasers from customers who had
# already made a repeat purchase within the first 60 days

repeat_timing = (
    customer_value_90d
    .merge(
        value_reconciliation[
            ["customer_unique_id", "has_subsequent_delivered_value"]
        ],
        on="customer_unique_id",
        how="left",
        validate="one_to_one"
    )
)

repeat_timing["generated_value_days_61_to_90"] = (
    repeat_timing["incremental_value_days_61_to_90"] > 0
)

repeat_timing["repeat_timing_group"] = np.select(
    [
        repeat_timing["generated_value_days_61_to_90"]
        & ~repeat_timing["has_subsequent_delivered_value"],

        repeat_timing["generated_value_days_61_to_90"]
        & repeat_timing["has_subsequent_delivered_value"],

        ~repeat_timing["generated_value_days_61_to_90"]
    ],
    [
        "First subsequent purchase during days 61–90",
        "Already repeated within 60 days",
        "No additional value during days 61–90"
    ],
    default="Unclassified"
)

repeat_timing_summary = (
    repeat_timing
    .groupby("repeat_timing_group", as_index=False)
    .agg(
        customers=("customer_unique_id", "count"),
        additional_value=(
            "incremental_value_days_61_to_90", "sum"
        )
    )
)

repeat_timing_summary["average_additional_value"] = (
    repeat_timing_summary["additional_value"]
    / repeat_timing_summary["customers"]
)

assert repeat_timing_summary["customers"].sum() == 71_988

assert np.isclose(
    repeat_timing_summary["additional_value"].sum(),
    27816.64,
    atol=0.01
)

display(repeat_timing_summary.round(2))

,repeat_timing_group,customers,additional_value,average_additional_value
0,Already repeated within 60 days,20,2538.66,126.93
1,First subsequent purchase during days 61–90,180,25277.98,140.43
2,No additional value during days 61–90,71788,0.00,0.00


In [22]:
# Compare historical merchandise value across three horizons
# using the same customers eligible for 180 days of observation

eligible_180d_ids = set(
    historical_intelligence.loc[
        historical_intelligence["order_purchase_timestamp"]
        + pd.Timedelta(days=210)
        <= observation_cutoff,
        "customer_unique_id"
    ]
)

eligible_180d_orders = customer_value_orders.loc[
    customer_value_orders["customer_unique_id"].isin(eligible_180d_ids)
].copy()

# Calculate merchandise value for each observation horizon
for horizon in [60, 90, 180]:
    eligible_180d_orders[f"value_{horizon}d"] = np.where(
        eligible_180d_orders["days_since_first_purchase"].between(
            0, horizon, inclusive="both"
        ),
        eligible_180d_orders["product_revenue"],
        0.0
    )

# Aggregate to one record per customer
customer_value_180d = (
    eligible_180d_orders
    .groupby("customer_unique_id", as_index=False)
    .agg(
        value_60d=("value_60d", "sum"),
        value_90d=("value_90d", "sum"),
        value_180d=("value_180d", "sum")
    )
)

# Isolate additional merchandise value by period
customer_value_180d["additional_value_days_61_to_90"] = (
    customer_value_180d["value_90d"]
    - customer_value_180d["value_60d"]
)

customer_value_180d["additional_value_days_91_to_180"] = (
    customer_value_180d["value_180d"]
    - customer_value_180d["value_90d"]
)

# Validate population, uniqueness and cumulative values
assert len(customer_value_180d) == 51_461
assert customer_value_180d["customer_unique_id"].is_unique

assert (
    customer_value_180d["value_60d"]
    <= customer_value_180d["value_90d"]
).all()

assert (
    customer_value_180d["value_90d"]
    <= customer_value_180d["value_180d"]
).all()

# Summarise results using the same eligible customer population
horizon_180d_comparison = pd.DataFrame({
    "Horizon": ["60 days", "90 days", "180 days"],
    "Eligible Customers": [len(customer_value_180d)] * 3,
    "Total Merchandise Value (R$)": [
        customer_value_180d["value_60d"].sum(),
        customer_value_180d["value_90d"].sum(),
        customer_value_180d["value_180d"].sum()
    ],
    "Average Value per Customer (R$)": [
        customer_value_180d["value_60d"].mean(),
        customer_value_180d["value_90d"].mean(),
        customer_value_180d["value_180d"].mean()
    ]
})

display(horizon_180d_comparison.round(2))

print("\nADDITIONAL MERCHANDISE VALUE BY PERIOD")

display(
    pd.DataFrame({
        "Period": ["Days 61–90", "Days 91–180"],
        "Additional Value (R$)": [
            customer_value_180d["additional_value_days_61_to_90"].sum(),
            customer_value_180d["additional_value_days_91_to_180"].sum()
        ],
        "Customers Generating Additional Value": [
            (
                customer_value_180d["additional_value_days_61_to_90"] > 0
            ).sum(),
            (
                customer_value_180d["additional_value_days_91_to_180"] > 0
            ).sum()
        ]
    }).round(2)
)

,Horizon,Eligible Customers,Total Merchandise Value (R$),Average Value per Customer (R$)
0,60 days,51461,6926323.84,134.59
1,90 days,51461,6947477.93,135.00
2,180 days,51461,6995949.65,135.95



ADDITIONAL MERCHANDISE VALUE BY PERIOD


,Period,Additional Value (R$),Customers Generating Additional Value
0,Days 61–90,21154.09,155
1,Days 91–180,48471.72,365


In [23]:
# Measure observed merchandise value over 60, 90, 180 and 365 days
# using the same customers eligible for the full 365-day horizon

eligible_365d_ids = set(
    historical_intelligence.loc[
        historical_intelligence["order_purchase_timestamp"]
        + pd.Timedelta(days=395)
        <= observation_cutoff,
        "customer_unique_id"
    ]
)

eligible_365d_orders = customer_value_orders.loc[
    customer_value_orders["customer_unique_id"].isin(eligible_365d_ids)
].copy()

# Calculate cumulative merchandise value at each horizon
for horizon in [60, 90, 180, 365]:
    eligible_365d_orders[f"value_{horizon}d"] = np.where(
        eligible_365d_orders["days_since_first_purchase"].between(
            0, horizon, inclusive="both"
        ),
        eligible_365d_orders["product_revenue"],
        0.0
    )

customer_value_365d = (
    eligible_365d_orders
    .groupby("customer_unique_id", as_index=False)
    .agg(
        value_60d=("value_60d", "sum"),
        value_90d=("value_90d", "sum"),
        value_180d=("value_180d", "sum"),
        value_365d=("value_365d", "sum")
    )
)

# Validate population and cumulative value consistency
assert len(customer_value_365d) == 18_606
assert customer_value_365d["customer_unique_id"].is_unique

for earlier, later in [(60, 90), (90, 180), (180, 365)]:
    assert (
        customer_value_365d[f"value_{earlier}d"]
        <= customer_value_365d[f"value_{later}d"]
    ).all()

# Compare the four horizons using exactly the same customer population
horizon_365d_summary = pd.DataFrame([
    {
        "Horizon": f"{horizon} days",
        "Eligible Customers": len(customer_value_365d),
        "Total Merchandise Value (R$)": (
            customer_value_365d[f"value_{horizon}d"].sum()
        ),
        "Average Value per Customer (R$)": (
            customer_value_365d[f"value_{horizon}d"].mean()
        )
    }
    for horizon in [60, 90, 180, 365]
])

display(horizon_365d_summary.round(2))

# Identify how much additional value accumulated in each later period
later_periods = [
    ("Days 61–90", 60, 90),
    ("Days 91–180", 90, 180),
    ("Days 181–365", 180, 365)
]

additional_value_summary = pd.DataFrame([
    {
        "Period": label,
        "Additional Value (R$)": (
            customer_value_365d[f"value_{end}d"]
            - customer_value_365d[f"value_{start}d"]
        ).sum(),
        "Customers Generating Additional Value": (
            (
                customer_value_365d[f"value_{end}d"]
                - customer_value_365d[f"value_{start}d"]
            ) > 0
        ).sum()
    }
    for label, start, end in later_periods
])

display(additional_value_summary.round(2))

,Horizon,Eligible Customers,Total Merchandise Value (R$),Average Value per Customer (R$)
0,60 days,18606,2493673.21,134.03
1,90 days,18606,2503440.41,134.55
2,180 days,18606,2523214.61,135.61
3,365 days,18606,2556174.75,137.38


,Period,Additional Value (R$),Customers Generating Additional Value
0,Days 61–90,9767.20,60
1,Days 91–180,19774.20,142
2,Days 181–365,32960.14,244


In [24]:
# Assess concentration of observed additional merchandise value
# within the 365-day eligible customer population

first_year_value = customer_value_365d.copy()

# Calculate merchandise value generated after the first 60 days
first_year_value["additional_value_after_60d"] = (
    first_year_value["value_365d"]
    - first_year_value["value_60d"]
)

# Identify customers generating additional value
returning_after_60d = first_year_value.loc[
    first_year_value["additional_value_after_60d"] > 0
].copy()

returning_after_60d = returning_after_60d.sort_values(
    "additional_value_after_60d",
    ascending=False
)

# Calculate concentration among customers with positive additional value
total_additional_value = (
    returning_after_60d["additional_value_after_60d"].sum()
)

concentration_results = []

for top_share in [0.10, 0.20, 0.50, 1.00]:
    customer_count = int(
        np.ceil(len(returning_after_60d) * top_share)
    )

    top_value = (
        returning_after_60d["additional_value_after_60d"]
        .head(customer_count)
        .sum()
    )

    concentration_results.append({
        "Top Share of Returning Customers (%)": top_share * 100,
        "Customers": customer_count,
        "Share of Additional Value (%)": (
            100 * top_value / total_additional_value
            if total_additional_value > 0
            else np.nan
        )
    })

concentration_summary = pd.DataFrame(concentration_results)

print("FIRST-YEAR ADDITIONAL VALUE CONCENTRATION")

print(f"Eligible customers: {len(first_year_value):,}")

print(
    "Customers generating value after day 60:",
    f"{len(returning_after_60d):,}"
)

print(
    "Total additional value after day 60:",
    f"R${total_additional_value:,.2f}"
)

display(concentration_summary.round(2))

FIRST-YEAR ADDITIONAL VALUE CONCENTRATION
Eligible customers: 18,606
Customers generating value after day 60: 426
Total additional value after day 60: R$62,501.54


,Top Share of Returning Customers (%),Customers,Share of Additional Value (%)
0,10.0,43,38.57
1,20.0,86,55.32
2,50.0,213,82.34
3,100.0,426,100.00


In [25]:
# Examine the relationship between existing propensity segments
# and observed additional merchandise value after day 60

propensity_value_365d = (
    customer_value_365d
    .merge(
        historical_intelligence[
            [
                "customer_unique_id",
                "propensity_segment",
                "repeat_purchase_propensity",
                "first_order_item_value"
            ]
        ],
        on="customer_unique_id",
        how="left",
        validate="one_to_one"
    )
)

# Calculate observed merchandise value after day 60
propensity_value_365d["additional_value_after_60d"] = (
    propensity_value_365d["value_365d"]
    - propensity_value_365d["value_60d"]
)

propensity_value_365d["generated_additional_value"] = (
    propensity_value_365d["additional_value_after_60d"] > 0
)

# Summarise subsequent value across propensity segments
propensity_value_summary = (
    propensity_value_365d
    .groupby("propensity_segment", as_index=False)
    .agg(
        customers=("customer_unique_id", "count"),
        customers_with_additional_value=(
            "generated_additional_value", "sum"
        ),
        average_predicted_propensity=(
            "repeat_purchase_propensity", "mean"
        ),
        total_additional_value=(
            "additional_value_after_60d", "sum"
        ),
        average_additional_value=(
            "additional_value_after_60d", "mean"
        )
    )
)

propensity_value_summary["additional_value_rate_pct"] = (
    propensity_value_summary["customers_with_additional_value"]
    / propensity_value_summary["customers"] * 100
)

# Reconcile the total population and observed additional value
assert propensity_value_summary["customers"].sum() == 18_606

assert np.isclose(
    propensity_value_summary["total_additional_value"].sum(),
    62501.54,
    atol=0.01
)

display(
    propensity_value_summary.sort_values(
        "average_predicted_propensity",
        ascending=False
    ).round(3)
)

,propensity_segment,customers,customers_with_additional_value,average_predicted_propensity,total_additional_value,average_additional_value,additional_value_rate_pct
0,High Propensity,3767,98,0.041,12499.31,3.318,2.602
2,Medium Propensity,5259,144,0.023,19725.19,3.751,2.738
1,Low Propensity,9580,184,0.015,30277.04,3.160,1.921


In [26]:
# Audit the calendar composition of the 365-day eligible population
# to understand how representative it is of the original customer cohort

cohort_composition_365d = (
    propensity_value_365d
    .merge(
        historical_intelligence[
            ["customer_unique_id", "order_purchase_timestamp"]
        ],
        on="customer_unique_id",
        how="left",
        validate="one_to_one"
    )
)

cohort_composition_365d["first_purchase_month"] = (
    cohort_composition_365d["order_purchase_timestamp"]
    .dt.to_period("M")
    .astype(str)
)

monthly_composition = (
    cohort_composition_365d
    .groupby("first_purchase_month", as_index=False)
    .agg(
        customers=("customer_unique_id", "count"),
        customers_with_additional_value=(
            "generated_additional_value", "sum"
        ),
        total_additional_value=(
            "additional_value_after_60d", "sum"
        )
    )
)

monthly_composition["additional_value_rate_pct"] = (
    100
    * monthly_composition["customers_with_additional_value"]
    / monthly_composition["customers"]
)

print("365-DAY ELIGIBLE COHORT COMPOSITION")

display(monthly_composition.round(2))

print("\nCOHORT COVERAGE")

print(
    "Earliest first purchase:",
    cohort_composition_365d["order_purchase_timestamp"].min()
)

print(
    "Latest first purchase:",
    cohort_composition_365d["order_purchase_timestamp"].max()
)

print(
    "Customers:",
    f"{len(cohort_composition_365d):,}"
)

assert len(cohort_composition_365d) == 18_606
assert monthly_composition["customers"].sum() == 18_606

365-DAY ELIGIBLE COHORT COMPOSITION


,first_purchase_month,customers,customers_with_additional_value,total_additional_value,additional_value_rate_pct
0,2016-09,4,0,0.00,0.00
1,2016-10,321,3,487.99,0.93
2,2016-12,1,0,0.00,0.00
3,2017-01,762,16,1691.58,2.10
4,2017-02,1751,34,4052.80,1.94
5,2017-03,2635,59,9348.47,2.24
6,2017-04,2350,54,8857.70,2.30
7,2017-05,3594,89,14530.01,2.48
8,2017-06,3136,86,11960.28,2.74
9,2017-07,3893,84,11533.81,2.16



COHORT COVERAGE
Earliest first purchase: 2016-09-04 21:15:19
Latest first purchase: 2017-08-01 23:41:09
Customers: 18,606



## Historical Customer Value: Findings and Business Implications

### Business Question

How much merchandise value do customers generate after their first purchase, and does our existing repeat-purchase segmentation help distinguish their subsequent monetary value?

### Key Findings

**1. Initial purchasing dominates observed 60-day merchandise value.**

Across 78,100 eligible customers, total observed 60-day merchandise value was R$10,606,400.87. Subsequent purchases contributed R$211,295.19, representing 1.99% of that value.

**2. Repeat-purchase likelihood and monetary value describe different aspects of customer behaviour.**

The Lower Value / High Propensity profile had an observed repeat rate of 3.63%, compared with 1.19% for Higher Value / Low Propensity. However, repeat purchasers in the latter profile generated R$231.40 on average, compared with R$80.37 in the former.

**3. Additional customer value accumulates beyond 60 days, but growth is modest in the historical sample.**

Among 18,606 customers eligible for 365-day observation, average merchandise value increased from R$134.03 at day 60 to R$137.38 at day 365.

**4. Additional merchandise value is concentrated among a minority of returning customers.**

Only 426 of the 18,606 eligible customers generated additional merchandise value during days 61–365. Among those 426 customers, the top 86 accounted for 55.32% of subsequent value.

**5. Short-term propensity does not consistently rank longer-term monetary value.**

Within the 365-day eligible population, medium-propensity customers generated R$3.75 in additional merchandise value per customer during days 61–365, compared with R$3.32 for high-propensity customers and R$3.16 for low-propensity customers.

These results are descriptive and do not establish predictive performance for long-term customer value.

### Measurement and Interpretation Limitations

- Merchandise value is calculated from delivered-order item prices, excluding freight, and does not represent profit or net revenue.
- The 60-day analysis covers 78,100 eligible customers, while the 365-day analysis covers only 18,606 earlier customers.
- Fixed-horizon value measurements cannot capture purchases beyond the available observation period.
- Existing propensity scores predict repeat purchasing within 60 days, not future revenue or profitability.
- Some historical customers were used during model development, so these comparisons are not independent predictive validation.
- Customer groups ranked using observed future spending cannot be assumed identifiable at the point of first purchase.
- The analysis does not establish the incremental financial impact of retention interventions.

### Business Implication

The existing Customer Intelligence Platform provides useful historical measures of repeat purchasing, initial transaction value, and observed subsequent spending.

However, these measures should remain distinct. Repeat-purchase propensity alone is insufficient to describe long-term customer monetary value.

The next analytical opportunity is to establish a properly defined future-value prediction target and assess whether the available data supports reliable out-of-time validation before introducing predictive Customer Lifetime Value.

### Conclusion

Historical customer-value analysis strengthens the platform by connecting purchasing behaviour to measurable commercial outcomes.

The evidence supports further customer-value modelling, but not yet production use of predicted lifetime value or claims about the profitability of retention interventions.
